# Advanced Optional Assignment — Deploy a Final-Assignment Model as a Live App

*University of Athens, MSc in BIS-Analytics — "Python for Data Science, Machine Learning and Artificial Intelligence".*

> This assignment is **optional** and **does not contribute to your course grade**. It exists for students who want to pursue a career in Data Science / ML and want a portfolio piece that proves they can take a model from a notebook to a public, working URL.

Companion files in this folder:

- [`final_assignment_python.ipynb`](final_assignment_python.ipynb) — the **mandatory** final assignment. Read that first; this advanced assignment **builds on its output**.
- [`advanced_optional_example/`](advanced_optional_example/) — a **fully worked example** that you can run, read, and adapt. Live demo: [thanarg/heart-disease-classifier](https://huggingface.co/spaces/thanarg/heart-disease-classifier).

---

## 1. What this assignment is

Take **one** of the three models you built for the mandatory final assignment (your **classification** *or* **regression** notebook — leave clustering aside, since Lecture 09 already covers that pattern), and **deploy it as a public web app on Hugging Face Spaces**.

By the end you will have:

- A small, self-contained app folder (`app.py`, `requirements.txt`, a small bundled CSV, a saved model artifact).
- A live, public Hugging Face Space — a real URL anyone can open.
- The skill of **separating offline training from online inference**, which is what the difference between a notebook and a product looks like.

## What this assignment is *not*

- Not a refactor of all three of your final-assignment notebooks. Pick **one** model (one notebook) and ship it.
- Not a graded artefact. There is no rubric — the only judge is whether the URL works in incognito mode on someone else's machine.
- Not an MLOps project. We are not building Docker images, CI pipelines, model registries, or monitoring dashboards. One vertical slice, done well.

## Why bother

Because "I trained a model in a Jupyter notebook" and "I deployed a model that other people can hit" are **separated by exactly the skills below**, and recruiters know it. Putting a working HF Space in your LinkedIn / GitHub puts you ahead of most graduates.

---

## 2. The pattern: train-once, serve-many

The mental model is two scripts that talk through one file on disk.

```text
  train_and_save_model.ipynb           app.py
  ─────────────────────────────        ───────────────────────────
  load CSV, split, fit Pipeline   ──►  model.joblib  ──►  joblib.load(...)
  evaluate, joblib.dump(...)                              .predict(new_row)
  (run once, offline)                                     (runs on every user click)
```

Three rules follow from that picture:

1. **`app.py` never trains models. Period.** It loads `model.joblib` (and a small comparison CSV) and calls `.predict()`. Training, candidate comparison, and winner selection all happen offline in `train_and_save_model.ipynb`. (Lecture 09's KMeans app refits at startup because clustering is unsupervised on 200 rows — that demo is the *exception*, not the rule, and not the pattern this assignment asks for.)
2. **Save the whole `Pipeline`, not just the classifier.** If you scaled features in training, the *same fitted scaler* must transform user input at inference time. Wrapping `[scaler, model]` in `sklearn.pipeline.Pipeline` and `joblib.dump`-ing the pipeline guarantees that. This single decision prevents the most common production-ML bug.
3. **Bundle a small CSV with the app, do not let users upload one.** A 200 KB stratified sample is enough for the EDA / Train tabs to look real. User upload would force you to validate schemas and handle adversarial files — out of scope.

---

## 3. Walkthrough on the worked example (heart-disease classification)

Open the [`advanced_optional_example/`](advanced_optional_example/) folder side-by-side with this notebook. Everything below references files inside it. The example uses a 5,000-row stratified sample of the heart-disease dataset already shipped with Lectures 07–08.

### 3.1 Bundle a small representative sample

The free **CPU Basic** Space gives you 2 vCPU, 16 GB RAM, and 50 GB of *ephemeral* disk (see [Hugging Face Spaces → Hardware resources](https://huggingface.co/docs/hub/spaces-overview#hardware-resources)). The Hub repo itself is generous too — public storage is "best-effort" on the free tier, with a per-file recommendation of <200 GB and a hard cap of 500 GB (see [Hub → Storage limits → Repository limitations and recommendations](https://huggingface.co/docs/hub/storage-limits#repository-limitations-and-recommendations)). So the lecture CSV (31 MB / 630k rows) would *technically* fit — none of the hard limits stop you.

We sample anyway, for three reasons that matter more than the hard limits in practice:

1. **Faster cold-start.** A Space rebuilds from scratch on every push, and free Spaces *go to sleep when idle* and re-load all data on wake — so a smaller repo means a noticeably snappier first impression for the recruiter who clicks your link.
2. **Snappier UI.** Every Gradio callback that touches `df` scans the whole bundled CSV. 213 KB is instant; 31 MB is sluggish on 2 vCPU.
3. **Be a good citizen of free public storage.** "Best-effort" means the platform reserves the right to push back on accounts that park gratuitous data — keep your Space focused on what the demo actually needs.

**Result: `heart_disease_sample.csv` is ~213 KB.** Section 8 of `train_and_save_model.ipynb` shows how it was generated in a few lines of `train_test_split(..., stratify=...)`, preserving the original 55 / 45 class balance.

### 3.2 Train the pipeline and save the artifact

Open [`advanced_optional_example/train_and_save_model.ipynb`](advanced_optional_example/train_and_save_model.ipynb) and read it top-to-bottom. The shape:

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, random_state=0)),
])
pipe.fit(X_train, y_train)

joblib.dump({
    "pipeline":      pipe,
    "feature_names": list(X.columns),
    "target_name":   TARGET,
    "positive_label": "Presence",
    "negative_label": "Absence",
}, "model.joblib")
```

On the bundled sample this gets ~88 % accuracy / 0.86 F1 — strong enough that the Predict tab feels meaningful, simple enough that a beginner can read every line.

### 3.3 Wrap it as a Gradio app

Open [`advanced_optional_example/app.py`](advanced_optional_example/app.py). It has three tabs:

- **EDA** — descriptive stats + a feature-vs-target histogram (read from the bundled CSV).
- **Model Card** — a *static* comparison of the three candidates from `model_comparison.csv` plus the winner's name and a short justification. No `.fit()` happens here — the numbers were computed offline.
- **Predict** — algorithm dropdown (default = F1 winner) + sliders for one new patient → the selected pipeline (one of three loaded once at startup from `model.joblib`) → predict + class probability. Lets a recruiter verify the Model Card claim by trying borderline patients across all three pre-fitted pipelines.

Read the docstring at the top of `app.py`. Notice what is *not* there: no `from sklearn import …`, no `.fit()` calls anywhere. That is the deliberate signal — training lives in the notebook, serving lives in `app.py`, and they communicate through two files on disk (`model.joblib`, `model_comparison.csv`). That separation is the entire deployment lesson.

### 3.4 Test locally

From the example folder:

```bash
python app.py
```

Open `http://localhost:7860`. Click through all three tabs. Try a patient profile in **Predict** that you'd expect to be high-risk. Sanity-check the prediction.

### 3.5 Deploy to Hugging Face Spaces

One-time setup (already done in Lecture 09c — re-do only if your token expired):

1. Create / log into [huggingface.co](https://huggingface.co/).
2. Generate a **Write** access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Cache it locally with `hf auth login` (paste the token when prompted).

Then run the cell below — it creates the Space and uploads your files in one go. Same Option-A pattern as Lecture 09.

> **Replace `USERNAME` and `SPACE_NAME` before running.** And do **not** commit your token anywhere.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

USERNAME   = "YOUR_HF_USERNAME"          # <-- change
SPACE_NAME = "my-final-assignment-app"   # <-- change
APP_DIR    = Path("advanced_optional_example")  # or your own folder

api = HfApi()
api.create_repo(
    repo_id=f"{USERNAME}/{SPACE_NAME}",
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True,
)

for fname in ["app.py", "requirements.txt", "README.md",
              "heart_disease_sample.csv", "model.joblib",
              "model_comparison.csv"]:
    api.upload_file(
        path_or_fileobj=str(APP_DIR / fname),
        path_in_repo=fname,
        repo_id=f"{USERNAME}/{SPACE_NAME}",
        repo_type="space",
    )
    print(f"uploaded {fname}")

print(f"Live: https://huggingface.co/spaces/{USERNAME}/{SPACE_NAME}")

First build takes 1–3 minutes. If the Space goes red, click the **Logs** tab on the Space page — the error is almost always (a) a missing line in `requirements.txt`, (b) a wrong relative path inside `app.py`, or (c) a file that wasn't uploaded.

---

## 4. Now do it with your own dataset

The whole point of the worked example is that you **copy the folder, swap the dataset, and re-run**. Concrete checklist:

1. **Pick one** of your final-assignment models (classification *or* regression).
2. Duplicate `advanced_optional_example/` and rename it (e.g. `<lastname>_<initial>_app/`).
3. Replace `heart_disease_sample.csv` with a **small** representative sample of your dataset (≤ 2 MB; stratify on the target if it is classification).
4. Edit `train_and_save_model.ipynb`: change the target column, swap the algorithm to whichever you used in your assignment, re-run end-to-end. `model.joblib` is regenerated.
5. Edit the **Predict** tab in `app.py`: replace the slider list with your features and their realistic ranges. The EDA and Train tabs adapt automatically because they read from the bundled CSV.
6. Run `python app.py` locally and click through every tab.
7. Push to a fresh Space using the cell above.

## Acceptance criteria

Submit a working Space URL where all of the following are true:

- The Space loads in incognito mode without errors.
- All three tabs render and respond.
- The **Predict** tab shows a sensible prediction for at least one input combination.
- The bundled CSV in the Space is ≤ 2 MB.
- `app.py` contains **zero** `.fit()` calls and **zero** `from sklearn` imports — training stays in the notebook.
- The repository contains `app.py`, `requirements.txt`, `README.md`, the sample CSV, `model.joblib`, `model_comparison.csv`, and `train_and_save_model.ipynb`.
- `requirements.txt` lists **only** what your app needs (pandas, scikit-learn, gradio, plotly, joblib for the worked example) — *not* the course's top-level 270-line `requirements.txt`.
- The README lists the dataset's source and license.

---

## 5. Submission and credit

There is no formal submission portal — this is genuinely optional. To get credit:

1. Push your code to a personal **public GitHub repository**.
2. Send Thanasis (a) the GitHub repo URL and (b) the live Hugging Face Space URL.
3. If your Space works, Thanasis will **endorse the relevant skills on your LinkedIn** (scikit-learn, Gradio, Hugging Face, Python, model deployment).

There is no deadline. Submit any time before the end of the academic year — the sooner the better, since LinkedIn endorsements compound.

---

## 6. Common pitfalls (read once before you start)

- **Absolute paths kill deployments.** In `app.py`, never write `pd.read_csv("/home/me/...")`. Use:
  ```python
  from pathlib import Path
  SCRIPT_DIR = Path(__file__).resolve().parent
  df = pd.read_csv(SCRIPT_DIR / "my_sample.csv")
  ```
- **Re-fitting the scaler at inference time silently corrupts predictions.** That is exactly what `Pipeline + joblib` prevents. Do not be clever and "rebuild" the scaler from the user's input row — it has only one row, so its mean is the row itself.
- **Don't commit secrets.** No tokens in code. No `.env`. Use environment variables on the Space if you ever need a key (the worked example does not).
- **Keep `requirements.txt` tiny.** A bloated requirements file makes the Space cold-start slow and is a red flag in code review. Pin only what you import.
- **Do not use the course's top-level `requirements.txt`** for your Space — that's the maintainer's full dev environment (~270 packages). Your app needs five.
- **Test in incognito.** Once you think it works, open the Space URL in a private window. If it loads there, it loads for the recruiter too.

---

Good luck — and please share your Space URL even if you change your mind about LinkedIn. I love seeing what students build.

— *Thanasis*